Quote from GLoVe github under [`scr/README.md`](https://github.com/stanfordnlp/GloVe/tree/master/src)

   > To train your own GloVe vectors, first you'll need to prepare your corpus as a single text file with all words separated by one or more spaces or tabs. If your corpus has multiple documents, the documents (only) should be separated by new line characters. Cooccurrence contexts for words do not extend past newline characters. Once you create your corpus, you can train GloVe vectors using the following 4 tools. An example is included in demo.sh, which you can modify as necessary.

# Importing the Data

In [1]:
import pandas as pd
import numpy as np

In [10]:
import os
print(os.getcwd())

/Users/caden/st_david-s-beacon/website/scripts/word embeddings


In [ ]:
psalms_verses = pd.read_csv("../../data/csv/cleaned_psalm_verses.csv")
psalms_verses

/Users/caden/st_david-s-beacon/website/scripts/word embeddings


,tradition,text,psalm_num,verse_num,verse
0,Orthodox,Bible,1,1,Blessed is the man Who walks not in the counse...
1,Orthodox,Bible,1,2,But his will is in the law of the Lord And in ...
2,Orthodox,Bible,1,3,He shall be like a tree Planted by streams of ...
3,Orthodox,Bible,1,4,Not so are the ungodly not so But they are lik...
4,Orthodox,Bible,1,5,Therefore the ungodly shall not rise in the ju...
...,...,...,...,...,...
5000,Orthodox,Psalter,150,64,"Butter of kine, and milk of sheep, with fat of..."
5001,Orthodox,Psalter,150,65,"So Jacob ate, and was filled; and the beloved ..."
5002,Orthodox,Psalter,150,66,"They provoked Me to anger with strange gods, a..."
5003,Orthodox,Psalter,150,67,"They sacrificed unto demons, not to God; to go..."


In [11]:
# Grouped Psalms (Bible & Psalter)
psalms = pd.read_csv("../../data/csv/grouped_psalm.csv")
psalms

,Unnamed: 0,tradition,text,psalm_num,verse,cleaned_verse
0,0,Orthodox,Bible,1,Blessed is the man Who walks not in the counse...,blessed man walk counsel ungodly stand way sin...
1,1,Orthodox,Bible,2,Why do the nations rage And the people meditat...,nation rage people meditate vain thing king ea...
2,2,Orthodox,Bible,3,A psalm by David when he fled from the face of...,psalm david fled face son absalom olord afflic...
3,3,Orthodox,Bible,4,For the End in psalms an ode by David You hear...,end psalm ode david heard icalled god righteou...
4,4,Orthodox,Bible,5,For the End concerning the inheritance a psalm...,end concerning inheritance psalm david give ea...
...,...,...,...,...,...,...
296,296,Orthodox,Psalter,146,The Lord doth build up Jerusalem; He shall gat...,lord doth build jerusalem ; shall gather toget...
297,297,Orthodox,Psalter,147,"Praise the Lord, O Jerusalem; praise thy God, ...","praise lord , jerusalem ; praise thy god , zio..."
298,298,Orthodox,Psalter,148,Praise ye the Lord from the heavens; praise Hi...,praise ye lord heaven ; praise highest . prais...
299,299,Orthodox,Psalter,149,"Sing unto the Lord a new song, His praise is i...","sing unto lord new song , praise congregation ..."


In [ ]:
# Renaming the last two columns as it should be psalm
psalms = psalms.rename(columns={'verse':'psalm',"cleaned_verse": "cleaned_psalm"})
psalms

,Unnamed: 0,tradition,text,psalm_num,psalm,cleaned_psalm
0,0,Orthodox,Bible,1,Blessed is the man Who walks not in the counse...,blessed man walk counsel ungodly stand way sin...
1,1,Orthodox,Bible,2,Why do the nations rage And the people meditat...,nation rage people meditate vain thing king ea...
2,2,Orthodox,Bible,3,A psalm by David when he fled from the face of...,psalm david fled face son absalom olord afflic...
3,3,Orthodox,Bible,4,For the End in psalms an ode by David You hear...,end psalm ode david heard icalled god righteou...
4,4,Orthodox,Bible,5,For the End concerning the inheritance a psalm...,end concerning inheritance psalm david give ea...
...,...,...,...,...,...,...
296,296,Orthodox,Psalter,146,The Lord doth build up Jerusalem; He shall gat...,lord doth build jerusalem ; shall gather toget...
297,297,Orthodox,Psalter,147,"Praise the Lord, O Jerusalem; praise thy God, ...","praise lord , jerusalem ; praise thy god , zio..."
298,298,Orthodox,Psalter,148,Praise ye the Lord from the heavens; praise Hi...,praise ye lord heaven ; praise highest . prais...
299,299,Orthodox,Psalter,149,"Sing unto the Lord a new song, His praise is i...","sing unto lord new song , praise congregation ..."


# Converting Data to `txt` files. 

Based on the Github repo, we need to do the training on a single txt file. I am considering each psalm to be a single document. Therefore we need to take the column of **cleaned_verse** and combined them into a single tx file. Since adding the label of each document would get in the way, I am making 2 parallel files

**Corpus for GloVe - `corpus.txt`**
1. Blessed is the man who walks not in the counsel of the ungodly...
2. Why do the nations rage, and the people plot in vain...
3. Blessed is the man that walketh not in the counsel of the ungodly...
4. Why do the heathen rage, and the people imagine a vain thing...

**Psalm Index - `corpus_index.txt`**

| Line | Psalm   | Tradition |
|------|---------|-----------|
| 1    | Psalm 1 | Psalter   |
| 2    | Psalm 2 | Psalter   |
| 3    | Psalm 1 | Bible     |
| 4    | Psalm 2 | Bible     |



In [15]:
with open("corpus.txt", "w", encoding="utf-8") as corpus_file, \
     open("corpus_index.txt", "w", encoding="utf-8") as index_file:
    
    for line_number, row in enumerate(psalms.itertuples(index=False), start=1):
        # 1. Corpus: cleaned text, one Psalm per line
        corpus_file.write(str(row.cleaned_psalm).strip().replace("\n", " ") + "\n")
        
        # 2. Index file: line number → Psalm ## + tradition
        index_file.write(f"{line_number}\tPsalm {row.psalm_num}\t{row.text}\n")